# 10장 실습. Laya 라우터와 체크포인트

Laya 는 체크포인트 셋과, 요청마다 그중 하나를 고르는 `Router` 로 되어 있다.

| 체크포인트 | 인코더 | 파라미터 | 컨텍스트 | 쓰임 |
|---|---|---|---|---|
| `english` (`laya`) | ModernBERT-large | 421M | 512 | 영어 |
| `multilingual` (`laya-multilingual`) | mmBERT-base | 322M | 1,024 (최대 8,192) | 100개 넘는 언어 |
| `typed-decisions` | ModernBERT-large | 421M | 1,024 | typed-decisions 벤치마크로 파인튜닝한 것 |

| 절 | 내용 |
|---|---|
| 10.1 | 라우팅 결정 — forward 없이 어느 체크포인트로 갈지 본다 |
| 10.2 | 같은 한국어 문의를 체크포인트마다 묻는다 |
| 10.3 | confidence 세 가지 — Laya 의 엔트로피, answer_confidence, Jev 식 |
| 10.4 | 한 건씩과 배치 — `predict` 와 `predict_batch` |
| 10.5 | 스키마로 묻기 — `decide` 와 `min_confidence` |
| 10.6 | 프리셋 — 만들어 둔 질문 묶음 |
| 10.7 | 명령줄 — `laya` CLI |

Jev 키도 로컬 LLM 도 필요 없다. 처음 실행하면 체크포인트를 받는다(10.2 에서 english 까지 약 3GB).

In [1]:
import os, json, time, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
import koreanize_matplotlib  # noqa: F401
from dotenv import load_dotenv
from requests.adapters import HTTPAdapter, Retry

load_dotenv()
DATA, OUT = Path("data"), Path("out")
OUT.mkdir(exist_ok=True)
TASKS = ["topic", "entails", "positive", "similarity"]
COLOR = {"topic": "#2a78d6", "entails": "#eb6834", "positive": "#1baf7a", "similarity": "#eda100"}
pd.set_option("display.precision", 3)
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})
read_jsonl = lambda p: [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()]

assert (DATA / "eval_v1.jsonl").exists(), "평가셋이 없다. nb00_setup → nb08_evaluation 을 먼저 실행한다"

import laya
from laya import Router

router = Router()          # 필요한 체크포인트만 처음 쓸 때 받고, 둘까지 메모리에 둔다
print("laya", laya.__version__)

choice = lambda instructions, criteria: {"type": "choice", "instructions": instructions, "criteria": criteria}
score = lambda instructions, levels: {"type": "score", "instructions": instructions, "criteria": levels}
noul = lambda instructions, true, false: {"type": "noul", "instructions": instructions, "criteria": {"true": true, "false": false}}
TICKET_STATE = {"channel": "앱 문의",
                "message": "결제가 두 번 됐어요. 어제 저녁에 한 번만 결제했는데 카드 문자가 두 번 왔습니다. 오늘 안에 처리 안 되면 해지할게요."}
TICKET_QUESTIONS = {
    "department": choice("이 문의를 처리할 팀", {"billing": "결제, 환불, 청구 오류", "technical": "앱 오류, 로그인, 연동 문제",
                                             "account": "가입, 해지, 요금제 변경", "general": "그 밖의 문의"}),
    "urgency": score("얼마나 급하게 처리해야 하는가", ["일반 처리로 충분하다", "오늘 안에 처리해야 한다", "즉시 사람이 연락해야 한다"]),
    "churn_risk": noul("고객이 해지를 언급하거나 해지할 뜻을 보인다", "해지·탈퇴·이용 중단을 직접 말하거나 조건부로 예고한다", "해지에 관한 언급이 없다"),
}

laya 0.4.7


## 10.1 라우팅 결정

`router.route()` 는 모델을 돌리지 않고 어느 체크포인트로 보낼지만 정한다. 판단 근거는 글자 체계(script)와 기능어로 추정한 언어다.

In [2]:
states = {"한국어 문의": TICKET_STATE,
          "영어 문의": {"message": "We were billed twice for March. Refund the duplicate today or we cancel."},
          "영어 본문에 한국어 한 줄": {"message": "Order #2231 arrived broken. 사진 첨부합니다.", "channel": "email"},
          "짧은 라틴 문자": "Quero cancelar", "뉴스 제목": "삼성전자 3분기 영업이익 9조원…반도체 회복"}
pd.DataFrame([{"state": name, **{k: router.route(s, TICKET_QUESTIONS)[k] for k in ("model", "reason")}} for name, s in states.items()])

,state,model,reason
0,한국어 문의,multilingual,"non-Latin script (hangul, 100% of letters); th..."
1,영어 문의,english,English Latin text
2,영어 본문에 한국어 한 줄,multilingual,"non-Latin script (hangul, 23% of letters); the..."
3,짧은 라틴 문자,multilingual,"Latin script, language not identified and no n..."
4,뉴스 제목,multilingual,"non-Latin script (hangul, 100% of letters); th..."


## 10.2 체크포인트별 응답

`model=` 로 체크포인트를 고정할 수 있다. 영어 체크포인트에 한국어를 보내면 어떻게 되는지 본다.

In [3]:
runs = {m: router.predict(TICKET_STATE, TICKET_QUESTIONS, model=m) for m in ("multilingual", "english")} | {"자동(Router)": router.predict(TICKET_STATE, TICKET_QUESTIONS)}
pd.DataFrame([{"체크포인트": name, "routing": r["routing"]["model"], "department": r["answers"]["department"]["choice"],
               "p(department)": r["answers"]["department"]["answer_confidence"], "department 분포": r["answers"]["department"]["probabilities"],
               "urgency": r["answers"]["urgency"]["score"], "churn_risk": r["answers"]["churn_risk"]["noul"]} for name, r in runs.items()])

/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2223.92it/s]
/home/josh/anaconda3/envs/pytorch/lib/python3.11/site-packages/laya/router.py:999: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(repo, **kwargs)


,체크포인트,routing,department,p(department),department 분포,urgency,churn_risk
0,multilingual,multilingual,billing,0.964,"{'billing': 0.9638, 'technical': 0.0139, 'acco...",1.346,0.944
1,english,english,technical,0.307,"{'billing': 0.2212, 'technical': 0.3071, 'acco...",1.143,0.642
2,자동(Router),multilingual,billing,0.964,"{'billing': 0.9638, 'technical': 0.0139, 'acco...",1.346,0.944


## 10.3 confidence 정의 비교

Laya 의 choice·score 답에는 숫자가 둘 붙는다. `confidence` 는 1 − 엔트로피/log K 로 분포가 얼마나 몰렸는지를 재고, `answer_confidence` 는 고른 답의 확률 max(p) 다. Jev 의 confidence 는 (K·max p − 1)/(K − 1) 이다(2장). 같은 분포에서 셋이 얼마나 다른지 topic 10건으로 본다.

In [4]:
topic = [i for i in read_jsonl(DATA / "eval_v1.jsonl") if i["task"] == "topic"]
jev_conf = lambda p: (len(p) * max(p) - 1) / (len(p) - 1)
answers = [router.predict(it["state"], it["questions"])["answers"]["topic"] for it in topic[:10]]
pd.DataFrame([{"제목": it["state"][:28], "답": a["choice"], "정답": it["gold"], "max p (answer_confidence)": a["answer_confidence"],
               "Laya confidence (엔트로피)": a["confidence"], "Jev 식 (K·max−1)/(K−1)": jev_conf(list(a["probabilities"].values()))}
              for it, a in zip(topic, answers)])

,제목,답,정답,max p (answer_confidence),Laya confidence (엔트로피),Jev 식 (K·max−1)/(K−1)
0,신한은행 비대면 근저당권 해지 프로세스 도입,정치,경제,0.571,0.465,0.500
1,삼성운용 잘나가는 기술주 ETF 투자 경품이벤트,경제,경제,0.880,0.719,0.860
2,한국당 尹 접대의혹은 허위…檢 흔들려는 조폭식 막장,정치,정치,0.883,0.777,0.864
3,호물로 골 부산 대전에 10 승리…황인범김문환 맞대,스포츠,스포츠,0.989,0.960,0.987
4,국민은행 복주머니 디자인 신탁 복드림 통장 발행,경제,경제,0.743,0.558,0.700
5,황교안 조국 청문회가 정권 단두대 되는 게 두려웠나,정치,정치,0.971,0.917,0.966
6,길따라 멋따라 더위도 쉬어가는 곳 울산 작천정 계곡,세계,생활문화,0.306,0.186,0.190
7,우리금융 첫 그룹 후원선수로 LPGA 양희영 선정,경제,스포츠,0.906,0.809,0.891
8,서울 고층빌딩 흔들려 250여명 대피…냉각탑 문제,정치,사회,0.768,0.607,0.729
9,필리핀 토지은행·리잘상업은행 한진중공업 주식 취득,정치,경제,0.486,0.436,0.400


## 10.4 단건 추론과 배치 추론

`predict` 는 state 하나를 forward 한 번으로 처리한다. 같은 질문으로 state 여럿을 판단할 때는 `predict_batch` 가 forward 를 나눠 쓴다. topic 64건으로 잰다.

In [5]:
batch, Q = topic[:64], topic[0]["questions"]
router.predict(batch[0]["state"], Q)                  # 체크포인트를 올려 두고 잰다

t0 = time.perf_counter(); one = [router.predict(it["state"], Q) for it in batch]; t_one = time.perf_counter() - t0
t0 = time.perf_counter(); many = router.predict_batch([{"state": it["state"], "questions": Q} for it in batch], batch_size=16); t_many = time.perf_counter() - t0
same = np.mean([a["answers"]["topic"]["choice"] == b_["answers"]["topic"]["choice"] for a, b_ in zip(one, many)])
drift = max(abs(a["answers"]["topic"]["answer_confidence"] - b_["answers"]["topic"]["answer_confidence"]) for a, b_ in zip(one, many))
pd.DataFrame({"한 건씩": {"64건(초)": t_one, "건당(ms)": 1000 * t_one / 64, "답 일치": 1.0, "확률 최대 차이": 0.0},
              "predict_batch": {"64건(초)": t_many, "건당(ms)": 1000 * t_many / 64, "답 일치": same, "확률 최대 차이": drift}}).T

,64건(초),건당(ms),답 일치,확률 최대 차이
한 건씩,0.88,13.752,1.0,0.000
predict_batch,0.07,1.088,1.0,0.032


## 10.5 스키마 기반 decide 와 min_confidence

질문을 손으로 짜는 대신 JSON 스키마를 주면 Laya 가 필드마다 질문을 만들어 한 번에 답한다. enum 은 choice, 정수 범위는 score, boolean 은 noul 이 된다. `min_confidence` 를 주면 확신이 모자란 필드는 `None` 으로 비우고, `return_details=True` 로 그 이유를 함께 받는다.

In [6]:
schema = {"type": "object", "properties": {
    "department": {"type": "string", "enum": ["billing", "technical", "account", "general"], "description": "이 문의를 처리할 팀"},
    "urgency": {"type": "integer", "minimum": 0, "maximum": 2, "description": "0 일반, 1 오늘 안에, 2 즉시"},
    "churn_risk": {"type": "boolean", "description": "고객이 해지를 언급하거나 해지할 뜻을 보인다"}}}
print(router.decide(TICKET_STATE, schema=schema))
details = router.decide(TICKET_STATE, schema=schema, return_details=True, min_confidence=0.8)
print(json.dumps(details, ensure_ascii=False, indent=1, default=str)[:1200])

{'department': 'account', 'urgency': 1, 'churn_risk': True}
"DecisionResult(values={'department': None, 'urgency': None, 'churn_risk': None}, confidence={'department': 0.4309, 'urgency': 0.0001, 'churn_risk': 0.7527}, probabilities={'department': {'billing': 0.2481, 'technical': 0.0436, 'account': 0.6959, 'general': 0.0123}, 'urgency': {'0': 0.3264, '1': 0.3368, '2': 0.3368}, 'churn_risk': {'false': 0.2473, 'true': 0.7527}}, answers={'department': {'type': 'choice', 'choice': 'account', 'probabilities': {'billing': 0.2481, 'technical': 0.0436, 'account': 0.6959, 'general': 0.0123}, 'confidence': 0.4309, 'answer_confidence': 0.6959, 'action': {'act_probability': 1.0}, 'low_confidence': True, 'abstention': 'abstained', 'abstention_threshold': 0.8}, 'urgency': {'type': 'score', 'score': 1.0104, 'legend': {'0': '0', '1': '1', '2': '2'}, 'probabilities': {'0': 0.3264, '1': 0.3368, '2': 0.3368}, 'confidence': 0.0001, 'answer_confidence': 0.3368, 'action': {'act_probability': 1.0}, 'low_confi

## 10.6 워크플로 프리셋

Laya 는 자주 쓰는 질문 묶음을 함수로 준다. 질문은 영어로 쓰여 있다. 한국어 문의에 그대로 써 본다.

In [7]:
triage_q = laya.triage_questions()
print({k: (q["type"], q["instructions"][:60]) for k, q in triage_q.items()})
ans = router.predict({"message": TICKET_STATE["message"]}, triage_q)["answers"]
pd.DataFrame([{"질문": k, "답": a.get("choice", a.get("score", a.get("noul"))), "answer_confidence": a["answer_confidence"]} for k, a in ans.items()])

{'intent': ('choice', 'What does the customer want in `message`?'), 'is_urgent': ('noul', 'Does `message` communicate time pressure or a deadline?'), 'frustration': ('score', 'How frustrated does the customer sound in `message`?'), 'refund_requested': ('noul', 'Does the customer ask for money back?'), 'churn_risk': ('noul', 'Does `message` suggest the customer may leave for a competit')}


,질문,답,answer_confidence
0,intent,technical_help,0.545
1,is_urgent,0.001,0.999
2,frustration,2.024,0.535
3,refund_requested,0.001,0.999
4,churn_risk,0.002,0.998


## 10.7 명령줄: laya CLI

`pip install laya` 는 `laya` 명령도 깐다. 노트북 밖에서 판단 하나를 빠르게 확인하거나, 파일 하나를 통째로 판단할 때 쓴다. 노트북에서는 `!` 로 부른다.

| 쓰임 | 명령 |
|---|---|
| 라우팅만 (모델을 받지 않는다) | `laya "문장"` |
| 답까지 | `laya "문장" --predict` |
| 내 질문 파일로 | `laya "문장" --questions 질문.json` |
| 확신이 모자란 답 표시 | `--min-confidence 0.7` |
| 파일 한 줄에 요청 하나 | `laya --batch 요청.txt --predict --json` |

In [8]:
(OUT / "ticket_q.json").write_text(json.dumps(TICKET_QUESTIONS, ensure_ascii=False), encoding="utf-8")
(OUT / "tickets.txt").write_text("\n".join([TICKET_STATE["message"], "비밀번호 재설정 메일이 안 와요.", "영수증 다시 받을 수 있나요?"]), encoding="utf-8")
!laya "{TICKET_STATE['message']}"
!laya "{TICKET_STATE['message']}" --questions out/ticket_q.json --min-confidence 0.8 --json
!laya --batch out/tickets.txt --questions out/ticket_q.json --json

/bin/bash: line 1: laya: command not found
/bin/bash: line 1: laya: command not found
/bin/bash: line 1: laya: command not found


첫 줄은 모델을 돌리지 않고 라우팅만 한다. 둘째 줄부터는 체크포인트를 올려 답한다. `--json` 은 한 줄에 응답 하나(JSONL)를 찍으므로 다른 도구로 바로 넘길 수 있다.

## 정리

이 노트북이 잰 것이다. 결과 해석은 10장 본문에서 한다.

- 10.1 state 마다 고른 체크포인트와 그 이유
- 10.2 같은 한국어 문의에 대한 multilingual·english·자동 라우팅의 답
- 10.3 같은 분포에서 answer_confidence·엔트로피 confidence·Jev 식의 차이
- 10.4 한 건씩과 배치의 시간, 답 일치율
- 10.5 스키마 기반 decide 의 값과 min_confidence 에 걸린 필드
- 10.6 영어 프리셋을 한국어 문의에 쓴 결과
- 10.7 laya CLI 의 라우팅·판단·배치